# Exercises XP Ninja: Guided Student Notebook

This guided notebook follows the **exercises on the platform**. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points are included only when a concept is important for intuition or transfer to other AI topics.


## Reference from the exercises

**What you will learn**  
- Implement state-of-the-art techniques to solve complex machine learning problems.
- Understand and apply advanced optimization methods for deep learning.
- Build and fine tune large scale models for real world applications.
- Explore cutting edge research areas like generative models and reinforcement learning.
- Develop strategies to handle imbalanced datasets and improve model robustness.

**What you will create**  
A deep learning model optimized using learning rate scheduling and gradient clipping. A generative model to create synthetic data. A reinforcement learning agent for a specific task. A robust model trained on an imbalanced dataset using SMOTE or focal loss. A comparison of performance before and after advanced strategies.


## Exercise 1: Advanced Optimization Techniques for Deep Learning

**As stated in the exercises**  
Objective. Improve model training stability and convergence using advanced optimization techniques.  
Instructions. Choose a deep learning model such as a CNN or RNN and a dataset such as CIFAR 10 or IMDB. Implement learning rate scheduling such as cosine annealing or step decay during training. Apply gradient clipping to prevent exploding gradients. Compare training stability and convergence with and without these techniques. Write a short analysis of how these techniques improve training.

**Guidance**  
Use CIFAR 10 with a small CNN for quick runs. Compare baseline vs scheduled learning rate and gradient clipping. Track training and validation curves.


**Learning point**  
Learning rate schedules reduce the step size as training progresses which helps convergence near minima. Cosine schedules can give sharper early progress. Gradient clipping caps update magnitude to prevent unstable jumps, especially in RNNs or deep CNNs.


In [3]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 767.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 159.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 108.8 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [4]:
# PREFILLED: just execute
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

# CIFAR-10 data
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
x_tr = x_tr.astype("float32")/255.0
x_te = x_te.astype("float32")/255.0
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)

x_tr.shape, y_tr_oh.shape, x_te.shape, y_te_oh.shape

In [4]:
# PREFILLED: just execute
def build_cnn(clipnorm=None, clipvalue=None, lr=1e-3):
    inputs = layers.Input(shape=(32,32,3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    opt = tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm, clipvalue=clipvalue)
    model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_cnn()
hist_base = baseline.fit(x_tr, y_tr_oh, validation_split=0.1, epochs=3, batch_size=128, verbose=2)

Epoch 1/3


ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data:0", shape=(None, 28, 28), dtype=float32) with name 'keras_tensor_8' and path ''. Expected shape (None, 28, 28, 3), but input has incompatible shape (None, 28, 28)[0m

Arguments received by Functional.call():
  • inputs=tf.Tensor(shape=(None, 28, 28), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
# To-Do: implement a learning rate schedule and train with gradient clipping
# Option 1: use tf.keras.optimizers.schedules.CosineDecay with an initial lr
# model_sched = build_cnn(lr=lr_schedule)
#
# Option 2: step decay via a callback
# def step_decay(epoch, lr):
#     return lr
# cb = tf.keras.callbacks.LearningRateScheduler(step_decay)
#
# Train a clipped model
# model_clip = build_cnn(clipnorm=1.0, lr=1e-3)
# hist_clip = model_clip.fit(...)
#
# Compare histories: plot loss and accuracy for baseline vs clipped or scheduled


**To-Do:** Plot two figures. Figure 1 shows training and validation accuracy per epoch for baseline and your improved run. Figure 2 shows training and validation loss per epoch. Explain in 3 to 5 sentences what changed.


## Exercise 2: Building a Generative Model

**As stated in the exercises**  
Objective. Create a generative model to produce synthetic data.  
Instructions. Choose an architecture such as a GAN or a VAE. Train on a dataset such as MNIST or CelebA. Evaluate sample quality using FID or visual inspection. Experiment with architectures such as DCGAN or loss choices. Write a short reflection on challenges and applications.

**Guidance**  
A VAE on MNIST is simpler to train than a GAN. Implement encoder and decoder with a reparameterization step. Use binary cross entropy reconstruction and KL regularization.


**Learning point**  
VAEs learn a latent distribution that supports interpolation and sampling. GANs can yield sharper images but are sensitive to instability and mode collapse.


In [5]:
# PREFILLED: just execute
from tensorflow.keras.datasets import mnist
(xm_tr, ym_tr), (xm_te, ym_te) = mnist.load_data()
xm_tr = xm_tr.astype("float32")/255.0
xm_te = xm_te.astype("float32")/255.0
xm_tr = np.expand_dims(xm_tr, -1)
xm_te = np.expand_dims(xm_te, -1)
xm_tr.shape, xm_te.shape

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


((60000, 28, 28, 1), (10000, 28, 28, 1))

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import keras.ops as ops # Ensure ops is imported for subsequent cells

latent_dim = 16

# Encoder
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
encoder = models.Model(encoder_inputs, [z_mean, z_log_var], name="encoder")
encoder.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 14, 14,    │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 7, 7, 64)  │     18,496 │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3136)      │          0 │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    401,536 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 16)        │      2,064 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 16)        │      2,064 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 424,480 (1.62 MB)

 Trainable params: 424,480 (1.62 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
latent_dim = 16

# Encoder
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
encoder = models.Model(encoder_inputs, [z_mean, z_log_var], name="encoder")
encoder.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 14, 14,    │        320 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 7, 7, 64)  │     18,496 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 3136)      │          0 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │    401,536 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 16)        │      2,064 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 16)        │      2,064 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 424,480 (1.62 MB)

 Trainable params: 424,480 (1.62 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Use the sampling layer to get the latent vector
z = Sampling()([z_mean, z_log_var])

In [9]:
# Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
decoder_res_shape = (7, 7, 64) # Calculate shape after conv layers in encoder
x = layers.Dense(int(np.prod(decoder_res_shape)), activation="relu")(latent_inputs)
x = layers.Reshape(decoder_res_shape)(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3136)           │        53,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 64)     │        36,928 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 32)     │        18,464 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 1)      │           289 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 108,993 (425.75 KB)

 Trainable params: 108,993 (425.75 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var = self.encoder(inputs)
        z = Sampling()([z_mean, z_log_var])
        reconstruction = self.decoder(z)

        # Reconstruction loss
        reconstruction_loss = ops.sum(
            tf.keras.losses.binary_crossentropy(inputs, reconstruction),
            axis=(1, 2) # Changed from (1, 2, 3) to (1, 2)
        )
        reconstruction_loss = ops.mean(reconstruction_loss) # Average over batch

        # KL divergence loss
        kl_loss = -0.5 * (1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
        kl_loss = ops.mean(ops.sum(kl_loss, axis=1)) # Sum over latent_dim, average over batch

        total_loss = reconstruction_loss + kl_loss
        self.add_loss(total_loss)
        return reconstruction

vae = VAE(encoder, decoder)
vae.compile(optimizer=tf.keras.optimizers.Adam())

In [21]:
# The loss calculation is now integrated into the VAE class's call method.
# This cell is no longer needed as the VAE is compiled within cell 6561a4bc.
# No code needed here.

In [11]:
vae.fit(xm_tr, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 35s 72ms/step - loss: 183.9120
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - loss: 120.2282
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 34s 72ms/step - loss: 110.4468
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - loss: 106.7081
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 33s 71ms/step - loss: 104.5457


**To-Do:** Display a grid of generated digits and comment on sample diversity and blur. If you choose a GAN instead, show a grid of samples and describe any instability you observed.


![image.png](https://github.com/user-attachments/assets/d63da82c-2d11-48cf-bd70-7cc68bd90722)


## Exercise 3: Handling Imbalanced Datasets

**As stated in the exercises**  
Objective. Improve performance on imbalanced datasets using advanced techniques.  
Instructions. Choose an imbalanced dataset such as credit card fraud. Apply SMOTE or ADASYN to balance the data. Train a model with focal loss or weighted cross entropy. Compare precision, recall, and F1 before and after. Write an analysis on the importance of handling imbalance.


**Learning point**  
Accuracy is misleading under class imbalance. Prefer precision, recall, F1, and PR AUC. SMOTE synthesizes minority points and focal loss focuses the objective on hard examples.


In [12]:
# PREFILLED: just execute
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_classification(n_samples=5000, n_features=20, n_informative=6, n_redundant=2,
                           weights=[0.96, 0.04], flip_y=0.005, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_trs = scaler.fit_transform(X_tr)
X_tes = scaler.transform(X_te)

print("Imbalance:", np.bincount(y_tr), "->", np.bincount(y_te))

Imbalance: [3834  166] -> [958  42]


In [13]:
# To-Do: apply SMOTE if available, else implement simple random oversampling of the minority
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=42)
    X_bal, y_bal = sm.fit_resample(X_trs, y_tr)
    print("After SMOTE:", np.bincount(y_bal))
except Exception as e:
    print("imblearn not available. Using naive oversampling.")
    idx_min = np.where(y_tr==1)[0]
    reps = int((len(y_tr)-len(idx_min)) / max(1, len(idx_min)))
    extra = np.random.choice(idx_min, size=reps*len(idx_min), replace=True)
    X_bal = np.vstack([X_trs, X_trs[extra]])
    y_bal = np.concatenate([y_tr, y_tr[extra]])
    print("After oversampling:", np.bincount(y_bal))

imblearn not available. Using naive oversampling.
After oversampling: [3834 3984]


In [18]:
# To-Do: train a small MLP with either class weights or focal loss
from tensorflow.keras import layers, models
def make_mlp():
    inputs = layers.Input(shape=(X_trs.shape[1],))
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    m = models.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m
#
# Option A: class weights
# ....
# print(classification_report(y_te, preds, digits=3))
#
# Option B: focal loss (implement function gamma=2)
def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
  return -alpha * (1 - y_pred) ** gamma * y_true * tf.math.log(y_pred) - (1 - alpha) * y_pred ** gamma * (1 - y_true) * tf.math.log(1 - y_pred)
m = make_mlp()
m.compile(optimizer="adam", loss=focal_loss, metrics=["accuracy"])
h = m.fit(X_bal, y_bal, validation_split=0.1, epochs=5, batch_size=256, verbose=2)
preds = (m.predict(X_tes, verbose=0).ravel() >= 0.5).astype(int)
print(classification_report(y_te, preds, digits=3))

Epoch 1/5
28/28 - 1s - 29ms/step - accuracy: 0.6231 - loss: 0.0657 - val_accuracy: 0.4642 - val_loss: 0.0612
Epoch 2/5
28/28 - 0s - 3ms/step - accuracy: 0.7470 - loss: 0.0465 - val_accuracy: 0.4859 - val_loss: 0.0563
Epoch 3/5
28/28 - 0s - 3ms/step - accuracy: 0.7719 - loss: 0.0399 - val_accuracy: 0.5435 - val_loss: 0.0474
Epoch 4/5
28/28 - 0s - 3ms/step - accuracy: 0.7982 - loss: 0.0349 - val_accuracy: 0.5767 - val_loss: 0.0441
Epoch 5/5
28/28 - 0s - 3ms/step - accuracy: 0.8258 - loss: 0.0308 - val_accuracy: 0.6867 - val_loss: 0.0351
              precision    recall  f1-score   support

           0      0.973     0.962     0.967       958
           1      0.308     0.381     0.340        42

    accuracy                          0.938      1000
   macro avg      0.640     0.672     0.654      1000
weighted avg      0.945     0.938     0.941      1000



In [19]:
confusion_matrix(y_te, preds)

array([[922,  36],
       [ 26,  16]])

**To-Do:** Plot a confusion matrix and compute precision, recall, and F1 before and after balancing. Explain in 3 to 5 sentences what changed and why.


## Exercise 4: Model Robustness and Adversarial Training

**As stated in the exercises**  
Objective. Improve robustness against adversarial attacks.  
Instructions. Train a model on MNIST or CIFAR 10. Generate adversarial examples using FGSM or PGD. Apply adversarial training by mixing some adversarial samples into batches. Evaluate robustness on clean and adversarial data. Write a short reflection on the importance and trade offs.


**Learning point**  
Adversarial training improves worst case robustness but can reduce clean accuracy. FGSM uses a single gradient step. PGD uses multiple steps and is stronger.


In [15]:
# PREFILLED: just execute
from tensorflow.keras.datasets import mnist
(xn_tr, yn_tr), (xn_te, yn_te) = mnist.load_data()
xn_tr = xn_tr.astype("float32")/255.0
xn_te = xn_te.astype("float32")/255.0
xn_tr = np.expand_dims(xn_tr, -1)
xn_te = np.expand_dims(xn_te, -1)

from tensorflow.keras.utils import to_categorical
yn_tr_oh = to_categorical(yn_tr, 10)
yn_te_oh = to_categorical(yn_te, 10)

def make_mnist_cnn():
    inputs = layers.Input(shape=(28,28,1))
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    m = models.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

model_clean = make_mnist_cnn()
hist_clean = model_clean.fit(xn_tr, yn_tr_oh, validation_split=0.1, epochs=2, batch_size=128, verbose=2)
clean_acc = float(model_clean.evaluate(xn_te, yn_te_oh, verbose=0)[1])
print({"clean_test_acc": clean_acc})

Epoch 1/2
422/422 - 22s - 52ms/step - accuracy: 0.9355 - loss: 0.2158 - val_accuracy: 0.9842 - val_loss: 0.0535
Epoch 2/2
422/422 - 21s - 50ms/step - accuracy: 0.9817 - loss: 0.0592 - val_accuracy: 0.9865 - val_loss: 0.0437
{'clean_test_acc': 0.9855999946594238}


In [17]:
# To-Do: implement FGSM and adversarial training
import tensorflow as tf

def fgsm(model, x, y_onehot, eps=0.2):
    # Ensure x is a TensorFlow tensor for gradient tape
    x = tf.convert_to_tensor(x)
    # use tf.GradientTape to get gradient of loss w.r.t inputs
    with tf.GradientTape() as tape:
        tape.watch(x)
        prediction = model(x)
        loss = tf.keras.losses.categorical_crossentropy(y_onehot, prediction)
    gradient = tape.gradient(loss, x)
    # x_adv = clip(x + eps * sign(grad), 0, 1)
    x_adv = x + eps * tf.sign(gradient)
    x_adv = tf.clip_by_value(x_adv, 0, 1)
    return x_adv

x_sample = xn_te[:128]
y_sample = yn_te_oh[:128]
x_adv = fgsm(model_clean, x_sample, y_sample, eps=0.2)
adv_loss, adv_acc = model_clean.evaluate(x_adv, y_sample, verbose=0)
print({"acc_on_adv_before_adv_training": float(adv_acc)})

# Now adversarial training for few epochs by mixing clean and FGSM examples
model_adv = make_mnist_cnn()
optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.CategoricalCrossentropy()

# Prepare dataset for adversarial training
batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((xn_tr, yn_tr_oh)).shuffle(10000).batch(batch_size)

for epoch in range(2):
    print(f"\nEpoch {epoch + 1}/2")
    for batch_idx, (x_batch, y_batch) in enumerate(dataset):
        # create a batch of adversarial examples on the fly from a subset
        x_adv_batch = fgsm(model_adv, x_batch, y_batch, eps=0.2)

        # train on a concatenation of clean and adversarial
        x_combined = tf.concat([x_batch, x_adv_batch], axis=0)
        y_combined = tf.concat([y_batch, y_batch], axis=0)

        with tf.GradientTape() as tape:
            predictions = model_adv(x_combined)
            loss = loss_fn(y_combined, predictions)
        gradients = tape.gradient(loss, model_adv.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_adv.trainable_variables))

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(dataset)} Loss: {loss.numpy():.4f}")

adv_clean = float(model_adv.evaluate(xn_te, yn_te_oh, verbose=0)[1])
x_adv_test = fgsm(model_adv, xn_te[:1000], yn_te_oh[:1000], eps=0.2)
adv_adv = float(model_adv.evaluate(x_adv_test, yn_te_oh[:1000], verbose=0)[1])
print({"adv_trained_clean_acc": adv_clean, "adv_trained_adv_acc": adv_adv})

{'acc_on_adv_before_adv_training': 0.3359375}

Epoch 1/2
  Batch 0/469 Loss: 2.3463
  Batch 100/469 Loss: 0.4270
  Batch 200/469 Loss: 0.2361
  Batch 300/469 Loss: 0.3536
  Batch 400/469 Loss: 0.2719

Epoch 2/2
  Batch 0/469 Loss: 0.2647
  Batch 100/469 Loss: 0.1506
  Batch 200/469 Loss: 0.2328
  Batch 300/469 Loss: 0.1014
  Batch 400/469 Loss: 0.2346
{'adv_trained_clean_acc': 0.9818999767303467, 'adv_trained_adv_acc': 0.8769999742507935}


**To-Do:** Show a few clean vs adversarial images side by side and comment on perceptibility. Reflect on the trade off between clean and robust accuracy.


![image.png](https://github.com/user-attachments/assets/44bebe22-845f-4d6e-b27b-2d687adf95f7)


## Conclusion

You applied advanced optimization, generative modeling, reinforcement learning, handling of class imbalance, and adversarial robustness. These exercises connect to real production concerns such as stability, data scarcity, safety, and fairness. Extend by adding cosine warm restarts, diffusion models, PPO with generalized advantage estimation, cost sensitive calibration on imbalanced data, and PGD adversarial training.
